# DistilBERT Fine-Tuning on AG News (Colab-ready)

This notebook fine-tunes **DistilBERT** on the AG News dataset and saves the model as `.safetensors` plus tokenizer files that you can download for deployment.

## Setup
Run the next cell to install dependencies in Colab.

In [ ]:
!pip install -q datasets transformers accelerate evaluate

## Imports and configuration

In [ ]:
import os
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)
import evaluate

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Reproducibility
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(seed)

## Load dataset

In [ ]:
# HuggingFace will cache the dataset in Colab's /root/.cache by default
dataset = load_dataset("ag_news")
label_names = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

print(dataset)
print("Label mapping:", label_names)

## Tokenization

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding=False,
        truncation=True,
        max_length=128
    )

# Map in batched mode for speed
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Train/validation split from the training set
split = tokenized_dataset["train"].train_test_split(test_size=0.1, seed=seed)
train_dataset = split["train"]
eval_dataset = split["test"]
test_dataset = tokenized_dataset["test"]

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_dataset)
print(eval_dataset)

## Define model and metrics

In [ ]:
num_labels = 4
model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
).to(device)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    results = {
        "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    }
    return results

## Training

In [ ]:
output_dir = "/content/model-output"

# Use a conservative TrainingArguments config that works with older transformers versions
base_args = dict(
    output_dir=output_dir,
    num_train_epochs=1,          # Increase for better performance (e.g., 3)
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    save_steps=500,              # adjust based on dataset size/epochs
    report_to="none",
    push_to_hub=False
)

# Add evaluation_strategy only if supported (older versions lack it)
if "evaluation_strategy" in TrainingArguments.__init__.__code__.co_varnames:
    base_args["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**base_args)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

train_result = trainer.train()
trainer.save_state()

print("Training complete.")

## Evaluation on test set

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=test_dataset)
print(test_metrics)

## Save model and tokenizer as `.safetensors`
This cell saves the fine-tuned model with safe serialization. The folder will contain `model.safetensors`, `config.json`, tokenizer files, and other metadata.

In [ ]:
export_dir = "/content/distilbert-ag-news"
os.makedirs(export_dir, exist_ok=True)

# Save model weights as safetensors
trainer.model.save_pretrained(export_dir, safe_serialization=True)
tokenizer.save_pretrained(export_dir)

print(f"Saved to {export_dir}")
print("Files:", os.listdir(export_dir))

## Download the artifacts
Run the next cell in Colab to download a zip containing the `.safetensors` model and tokenizer.

In [ ]:
import shutil
from google.colab import files

zip_path = "/content/distilbert-ag-news.zip"
shutil.make_archive(base_name=zip_path.replace('.zip', ''), format="zip", root_dir=export_dir)

print(f"Created archive: {zip_path}")
files.download(zip_path)